# Construire un mini-modèle de langage depuis zéro

Dans cet exercice, nous entraînons un petit Transformer à prédire le caractère suivant d'un texte. Ses poids commencent aléatoirement : aucun LLM pré-entraîné, aucun Qwen et aucun tokenizer pré-entraîné ne sont chargés.

Ce modèle minuscule sert uniquement à comprendre le mécanisme. Il n'a ni la taille, ni les données, ni les capacités de Qwen. Il peut mémoriser quelques motifs du texte et produire des suites maladroites.

Le code utilise PyTorch et peut s'exécuter sur CPU. L'entraînement proposé est court ; le modèle complet n'a que quelques dizaines de milliers de paramètres.

## Le principe

1. Le texte est découpé en caractères, et chaque caractère reçoit un identifiant.
2. À chaque position, le modèle regarde le caractère courant et les caractères précédents, jamais ceux du futur.
3. L'attention multi-tête combine plusieurs façons de relier les positions.
4. Le modèle prédit le caractère suivant. On compare sa prédiction au vrai caractère, puis la rétropropagation ajuste ses poids.
5. Après l'entraînement, on lui donne un début de texte et il prédit les caractères suivants un par un.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil utilisé :", device)
print("Version de PyTorch :", torch.__version__)

Appareil utilisé : cpu
Version de PyTorch : 2.14.1+cpu


## 1. Préparer un petit texte d'entraînement

Le corpus ci-dessous est écrit pour cet exercice. Pour avoir assez d'exemples, on le répète. Le modèle apprend uniquement à partir de ces caractères : il ne connaît pas le français comme un grand modèle.

In [3]:
story = (
    "Lina ouvre un vieux carnet. Dans ses pages, elle trouve une carte et une phrase : "
    "suis la rivière jusqu'au pont. Elle marche sous les arbres, écoute les oiseaux, "
    "puis découvre une petite maison. À l'intérieur, une lampe éclaire une table. "
    "Lina pose le carnet et écrit : chaque voyage commence par une question. "
)
text = story * 30

characters = sorted(set(text))
vocab_size = len(characters)
char_to_id = {character: index for index, character in enumerate(characters)}
id_to_char = {index: character for character, index in char_to_id.items()}

def encode(text_to_encode):
    return [char_to_id[character] for character in text_to_encode]

def decode(token_ids):
    return "".join(id_to_char[int(token_id)] for token_id in token_ids)

data = torch.tensor(encode(text), dtype=torch.long)
print("Longueur du corpus :", len(text), "caractères")
print("Taille du vocabulaire :", vocab_size, "caractères distincts")
print("Extrait décodé :", decode(data[:120]))

Longueur du corpus : 9330 caractères
Taille du vocabulaire : 33 caractères distincts
Extrait décodé : Lina ouvre un vieux carnet. Dans ses pages, elle trouve une carte et une phrase : suis la rivière jusqu'au pont. Elle ma


In [5]:
block_size = 64

def get_batch(batch_size=16):
    starts = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    inputs = torch.stack([data[start : start + block_size] for start in starts])
    targets = torch.stack([data[start + 1 : start + block_size + 1] for start in starts])
    return inputs.to(device), targets.to(device)

sample_x, sample_y = get_batch(batch_size=1)
print("Entrée :    ", decode(sample_x[0]))
print("Cible +1 : ", decode(sample_y[0]))
print("Chaque caractère de la cible est le caractère suivant de l'entrée.")

Entrée :     t une phrase : suis la rivière jusqu'au pont. Elle marche sous l
Cible +1 :   une phrase : suis la rivière jusqu'au pont. Elle marche sous le
Chaque caractère de la cible est le caractère suivant de l'entrée.


## 2. Construire le Transformer causal

La matrice triangulaire dans l'attention masque les positions futures. Ainsi, pour prédire un caractère, le modèle ne peut utiliser que le contexte déjà vu. Les projections Q, K et V sont des couches apprises pendant l'entraînement.

In [13]:
class CausalSelfAttention(nn.Module):
    def __init__(self, embedding_size, number_of_heads, block_size):
        super().__init__()
        if embedding_size % number_of_heads != 0:
            raise ValueError("La dimension doit être divisible par le nombre de têtes.")
        self.number_of_heads = number_of_heads
        self.head_size = embedding_size // number_of_heads
        self.qkv = nn.Linear(embedding_size, 3 * embedding_size, bias=False)
        self.projection = nn.Linear(embedding_size, embedding_size)
        causal_mask = torch.tril(torch.ones(block_size, block_size, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        batch_size, sequence_length, embedding_size = x.shape
        qkv = self.qkv(x).view(
            batch_size, sequence_length, 3, self.number_of_heads, self.head_size
        )
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        scores = (q @ k.transpose(-2, -1)) * (self.head_size ** -0.5)
        mask = self.causal_mask[:sequence_length, :sequence_length]
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        attended = weights @ v
        attended = attended.transpose(1, 2).contiguous().view(
            batch_size, sequence_length, embedding_size
        )
        return self.projection(attended)


class TransformerBlock(nn.Module):
    def __init__(self, embedding_size, number_of_heads, block_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(embedding_size)
        self.attention = CausalSelfAttention(embedding_size, number_of_heads, block_size)
        self.norm2 = nn.LayerNorm(embedding_size)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_size, 4 * embedding_size),
            nn.GELU(),
            nn.Linear(4 * embedding_size, embedding_size),
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class TinyLanguageModel(nn.Module):
    def __init__(
        self, vocab_size, block_size, embedding_size=48,
        number_of_heads=4, number_of_layers=2
    ):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, embedding_size)
        self.position_embedding = nn.Embedding(block_size, embedding_size)
        self.blocks = nn.Sequential(
            *[
                TransformerBlock(embedding_size, number_of_heads, block_size)
                for _ in range(number_of_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(embedding_size)
        self.language_head = nn.Linear(embedding_size, vocab_size)

    def forward(self, token_ids, targets=None):
        _, sequence_length = token_ids.shape
        positions = torch.arange(sequence_length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        x = self.blocks(x)
        logits = self.language_head(self.final_norm(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, token_ids, number_of_new_tokens, temperature=1.0):
        if temperature <= 0:
            raise ValueError("La température doit être strictement positive.")

        for _ in range(number_of_new_tokens):
            context = token_ids[:, -self.block_size:]
            logits, _ = self(context)
            next_character_logits = logits[:, -1, :] / temperature
            probabilities = F.softmax(next_character_logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)
            token_ids = torch.cat((token_ids, next_token), dim=1)

        return token_ids

## 3. Vérifier le modèle avant l'entraînement

Cette vérification lance une passe avant sur un mini-lot. Les paramètres PyTorch sont initialisés aléatoirement par défaut ; ici, aucun fichier de poids n'est téléchargé ou chargé. La forme des logits est `(lot, positions, caractères possibles)`.

In [14]:
model = TinyLanguageModel(vocab_size, block_size).to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
inputs, targets = get_batch(batch_size=2)
logits, initial_loss = model(inputs, targets)
print("Nombre de paramètres entraînables :", parameter_count)
print("Forme des prédictions :", tuple(logits.shape))
print("Perte initiale :", round(initial_loss.item(), 3))
assert logits.shape == (2, block_size, vocab_size)
assert all(parameter.requires_grad for parameter in model.parameters())
print("Vérification réussie : modèle créé depuis une initialisation aléatoire.")

Nombre de paramètres entraînables : 62625
Forme des prédictions : (2, 64, 33)
Perte initiale : 3.601
Vérification réussie : modèle créé depuis une initialisation aléatoire.


## 4. Entraîner à prédire le caractère suivant

À chaque itération, le modèle prédit tous les caractères de la séquence. La perte mesure l'écart entre ces prédictions et les vraies cibles. `backward()` calcule les gradients et `step()` met à jour les poids. Lance cette cellule pour l'entraînement court ; sur CPU, cela peut prendre un petit moment.

In [21]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
training_steps = 1000

model.train()
for step in range(training_steps):
    inputs, targets = get_batch(batch_size=16)
    _, loss = model(inputs, targets)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 100 == 0 or step == training_steps - 1:
        print(f"Étape {step + 1:4d}/{training_steps} | perte : {loss.item():.3f}")

Étape    1/1000 | perte : 1.066
Étape  101/1000 | perte : 0.382
Étape  201/1000 | perte : 0.175
Étape  301/1000 | perte : 0.097
Étape  401/1000 | perte : 0.070
Étape  501/1000 | perte : 0.063
Étape  601/1000 | perte : 0.067
Étape  701/1000 | perte : 0.064
Étape  801/1000 | perte : 0.068
Étape  901/1000 | perte : 0.070
Étape 1000/1000 | perte : 0.055


## 5. Générer un texte

Le modèle démarre avec un caractère du vocabulaire, puis prédit un caractère à la fois. Comme le corpus est petit et l'entraînement très court, le résultat sera souvent imparfait ou répétitif : c'est normal. Essaie différents débuts et observe l'effet du nombre de pas d'entraînement.

In [22]:
start_text = "ranis d"
start_ids = torch.tensor([encode(start_text)], dtype=torch.long, device=device)
generated_ids = model.generate(start_ids, number_of_new_tokens=300, temperature=0.8)
print(decode(generated_ids[0].cpu()))

ranis découvre une petite maison. À l'intérieur, une lampe éclaire une table. Lina pose le carnet et écrit : chaque voyage commence par une question. Lina ouvre un vieux carnet. Dans ses pages, elle trouve une carte et une phrase : suis la rivière jusqu'au pont. Elle marche sous les arbres, écoute les oise


## Ce que l'exercice montre

- Les poids du modèle commencent sans connaissance pré-entraînée et sont ajustés avec les exemples du corpus.
- L'attention causale empêche de regarder les caractères futurs pendant la prédiction.
- L'entraînement consiste ici à apprendre la tâche suivante : prédire le prochain caractère.
- Un vrai LLM comme Qwen suit des idées apparentées, mais utilise des tokens, une architecture et des jeux de données beaucoup plus grands, ainsi que des ressources de calcul considérables. Ce mini-modèle ne reproduit pas son entraînement ni ses capacités.